# Testing models

In [ ]:
import torch
import torch.nn as nn
from models.blocks import ConvNeXtcausal
from torch.nn.utils.parametrizations import weight_norm
#from transformers import EncodecModel
from utils.mel import MelSpectra

No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention


In [38]:
mel_extractor = MelSpectra(
    sample_rate=24000,
    n_fft=1024,
    hop_length=256,
    n_mels=128
)

In [ ]:
class EncoderFast(nn.Module):
    def __init__(self, in_channels: int, dim: int, latent_dim: int, 
                 inter_channels: int, num_blocks: int):
        super(EncoderFast, self).__init__()

        strides = [8, 8, 8] 

        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv1 = weight_norm(nn.Conv1d(in_channels, dim, kernel_size=7, padding=0))
        self.stages = nn.ModuleList()

        for s in strides:
            blocks = [ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)]
            stage = nn.Sequential(
                *blocks,
                nn.ConstantPad1d((s - 1, 0), 0),  # causal padding
                nn.Conv1d(dim, dim, kernel_size=s, stride=s, padding=0)
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.proj = nn.Linear(dim, latent_dim)

    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv1(x)                # (B, dim, T)

        for stage in self.stages:
            x = stage(x)                 # (B, dim, T/stride_total)

        x = x.transpose(1, 2)           # (B, T', dim)
        x = self.norm(x)
        x = self.proj(x)                 # (B, T', latent_dim)
        x = x.transpose(1, 2)           # (B, latent_dim, T')
        return x

In [50]:
audio = torch.randn(1, 1, 573)
encoder = EncoderFast(1, 128, 32, 256, 3)
emb = encoder(audio)
print(emb.shape)

torch.Size([1, 32, 2])


In [25]:
x = torch.ones(1, 1, 24000)
model = EncodecModel.from_pretrained('facebook/encodec_24khz')
with torch.no_grad():
    emb = model.encoder(x)
    #emb = (emb - emb.mean(dim=-1, keepdim=True)) / (emb.std(dim=-1, keepdim=True) + 1e-5)
print(f"Encoder output shape: {emb.shape}")
print(f"Encoder output mean: {emb.mean().item():.4f}, std: {emb.std().item():.4f}")

Loading weights: 100%|██████████| 252/252 [00:00<00:00, 3322.05it/s]


Encoder output shape: torch.Size([1, 128, 75])
Encoder output mean: -1.1315, std: 8.3578


In [33]:
import sys
import torch
import torchaudio
sys.path.append('../stable-audio-3')
from stable_audio_3 import AutoencoderModel

x, sr = torchaudio.load('/home/lois/wavenext/logs/05-06_at_03_27_42/wavenext/version_0/audio_epoch_80/sample_0_real.wav')
x = torchaudio.functional.resample(x, orig_freq=sr, new_freq=44100)
ae = AutoencoderModel.from_pretrained("same-s")
with torch.no_grad():
    emb = ae.encode(x, 44100)
print(f"Autoencoder output shape: {emb.shape}")

audio_out = ae.decode(emb)

from IPython.display import Audio
Audio(audio_out.cpu().squeeze(), rate=44100)

Autoencoder output shape: torch.Size([1, 256, 12])


In [38]:
x = torch.randn(1, 1, 4096)
with torch.no_grad():
    emb = ae.encode(x, 44100)
print(f"Autoencoder output shape: {emb.shape}")

Autoencoder output shape: torch.Size([1, 256, 2])


In [3]:
class Decoder(nn.Module):
    def __init__(self, in_channels: int, dim: int, shift_dim: int, inter_channels: int, num_blocks: int):
        super(Decoder, self).__init__()
        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv = nn.Conv1d(in_channels, dim, kernel_size=7, padding=0)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.blocks = nn.ModuleList([ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)])
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, shift_dim, bias=False) 
        # (B, shift_dim, T) -> (B, 1 , shift_dim * T)
    
    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv(x)
        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.norm(x)
        x = x.transpose(1, 2)  # (B, dim, T)

        for block in self.blocks:
            x = block(x)

        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.linear1(x)
        x = self.linear2(x) # (B, T, shift_dim)
        x = x.view(x.size(0), -1) # (B, shift_dim * T)

        return x

In [21]:
s_dim = x.size(-1) // emb.size(-1)
decoder = Decoder(in_channels=128, dim=512, shift_dim=s_dim, inter_channels=256, num_blocks=2)
decoder = decoder.to(emb.device) 
y = decoder(emb)
print(f"Decoder output shape: {y.shape}")

Decoder output shape: torch.Size([1, 24000])


In [26]:
with torch.no_grad():
    y = model.decoder(emb)

In [27]:
import torch.nn.functional as F
mel_original = mel_extractor(x)
mel_reconstructed = mel_extractor(y)
print(f"EnCodec native mel loss: {F.l1_loss(mel_reconstructed, mel_original).item():.4f}")

EnCodec native mel loss: 53.4766


In [9]:
from encodec import EncodecModel
from encodec.utils import convert_audio
import torchaudio

wav = torch.load('/home/lois/wavenext/logs/05-06_at_03_27_42/wavenext/version_0/audio_epoch_80/sample_0_real.wav')

model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav)  # liste de (codes, scale)
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))
    print(f"emb shape : {emb.shape}")  # [1, 128, 75]

IndexError: pop from empty list

In [5]:
import torch
import yaml
from models.wavenext_prior import WaveNeXtLatent
import torchaudio

def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

config = load_config('/home/lois/wavenext/config_prior_24k.yaml')

model = WaveNeXtLatent(
        dim=config['dim'],
        sample_rate=config['sample_rate'],
        fft_dim=config['fft_dim'],
        shift_dim=config['shift_dim'],
        n_mels=config['n_mels'],
        k=config['k'],
        lr_g=config['learning_rate_g'],
        lr_d=config['learning_rate_d'],
        prior=config['prior']
    ).to('cuda')

model.load_state_dict(torch.load("/home/lois/wavenext/checkpoints/05-06_at_03_27_42/wavenext-epoch=88-val_mel_loss=0.893.ckpt")['state_dict'])
model.eval()

/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


WaveNeXtLatent(
  (decoder): Decoder(
    (pad_input): ConstantPad1d(padding=(6, 0), value=0)
    (conv): Conv1d(128, 512, kernel_size=(7,), stride=(1,))
    (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
    (blocks): ModuleList(
      (0-7): 8 x ConvNeXtcausal(
        (pad): ConstantPad1d(padding=(6, 0), value=0)
        (depthwise): Conv1d(512, 512, kernel_size=(7,), stride=(1,), groups=512)
        (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        (pointwise1): Linear(in_features=512, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (pointwise2): Linear(in_features=1536, out_features=512, bias=True)
      )
    )
    (linear1): Linear(in_features=512, out_features=512, bias=True)
    (linear2): Linear(in_features=512, out_features=320, bias=False)
  )
  (discriminator_mpd): MPD(
    (discriminators): ModuleList(
      (0-4): 5 x OnePeriod(
        (conv): ModuleList(
          (0): ParametrizedConv2d(
            1, 3

In [19]:
from encodec import EncodecModel

wav = torchaudio.load('/home/lois/wavenext/logs/05-06_at_03_27_42/wavenext/version_0/audio_epoch_80/sample_0_real.wav')[0].unsqueeze(0).to('cuda')

model = EncodecModel.encodec_model_24khz().to('cuda')
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav) 
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))
    fake = model.decoder(emb.to('cuda'))

print(f"Encoder output shape: {emb.shape}")
from IPython.display import Audio
Audio(fake.cpu().squeeze(), rate=24000)


Encoder output shape: torch.Size([1, 128, 75])


In [10]:
import torch
import typing as tp
import torch.nn as nn
import numpy as np
import warnings
import math
from torch.nn.utils import spectral_norm, weight_norm
import einops
import torch.nn.functional as F

CONV_NORMALIZATIONS = frozenset(['none', 'weight_norm', 'spectral_norm',
                                 'time_layer_norm', 'layer_norm', 'time_group_norm'])

class ConvLayerNorm(nn.LayerNorm):
    """
    Convolution-friendly LayerNorm that moves channels to last dimensions
    before running the normalization and moves them back to original position right after.
    """
    def __init__(self, normalized_shape: tp.Union[int, tp.List[int], torch.Size], **kwargs):
        super().__init__(normalized_shape, **kwargs)

    def forward(self, x):
        x = einops.rearrange(x, 'b ... t -> b t ...')
        x = super().forward(x)
        x = einops.rearrange(x, 'b t ... -> b ... t')
        return

def apply_parametrization_norm(module: nn.Module, norm: str = 'none') -> nn.Module:
    assert norm in CONV_NORMALIZATIONS
    if norm == 'weight_norm':
        return weight_norm(module)
    elif norm == 'spectral_norm':
        return spectral_norm(module)
    else:
        # We already check was in CONV_NORMALIZATION, so any other choice
        # doesn't need reparametrization.
        return module

def get_norm_module(module: nn.Module, causal: bool = False, norm: str = 'none', **norm_kwargs) -> nn.Module:
    """Return the proper normalization module. If causal is True, this will ensure the returned
    module is causal, or return an error if the normalization doesn't support causal evaluation.
    """
    assert norm in CONV_NORMALIZATIONS
    if norm == 'layer_norm':
        assert isinstance(module, nn.modules.conv._ConvNd)
        return ConvLayerNorm(module.out_channels, **norm_kwargs)
    elif norm == 'time_group_norm':
        if causal:
            raise ValueError("GroupNorm doesn't support causal evaluation.")
        assert isinstance(module, nn.modules.conv._ConvNd)
        return nn.GroupNorm(1, module.out_channels, **norm_kwargs)
    else:
        return nn.Identity()


def pad1d(x: torch.Tensor, paddings: tp.Tuple[int, int], mode: str = 'zero', value: float = 0.):
    """Tiny wrapper around F.pad, just to allow for reflect padding on small input.
    If this is the case, we insert extra 0 padding to the right before the reflection happen.
    """
    length = x.shape[-1]
    padding_left, padding_right = paddings
    assert padding_left >= 0 and padding_right >= 0, (padding_left, padding_right)
    if mode == 'reflect':
        max_pad = max(padding_left, padding_right)
        extra_pad = 0
        if length <= max_pad:
            extra_pad = max_pad - length + 1
            x = F.pad(x, (0, extra_pad))
        padded = F.pad(x, paddings, mode, value)
        end = padded.shape[-1] - extra_pad
        return padded[..., :end]
    else:
        return F.pad(x, paddings, mode, value)

def get_extra_padding_for_conv1d(x: torch.Tensor, kernel_size: int, stride: int,
                                 padding_total: int = 0) -> int:
    """See `pad_for_conv1d`.
    """
    length = x.shape[-1]
    n_frames = (length - kernel_size + padding_total) / stride + 1
    ideal_length = (math.ceil(n_frames) - 1) * stride + (kernel_size - padding_total)
    return ideal_length - length


class NormConv1d(nn.Module):
    """Wrapper around Conv1d and normalization applied to this conv
    to provide a uniform interface across normalization approaches.
    """
    def __init__(self, *args, causal: bool = False, norm: str = 'none',
                 norm_kwargs: tp.Dict[str, tp.Any] = {}, **kwargs):
        super().__init__()
        self.conv = apply_parametrization_norm(nn.Conv1d(*args, **kwargs), norm)
        self.norm = get_norm_module(self.conv, causal, norm, **norm_kwargs)
        self.norm_type = norm

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        return x

class SLSTM(nn.Module):
    """
    LSTM without worrying about the hidden state, nor the layout of the data.
    Expects input as convolutional layout.
    """
    def __init__(self, dimension: int, num_layers: int = 2, skip: bool = True):
        super().__init__()
        self.skip = skip
        self.lstm = nn.LSTM(dimension, dimension, num_layers)

    def forward(self, x):
        x = x.permute(2, 0, 1)
        y, _ = self.lstm(x)
        if self.skip:
            y = y + x
        y = y.permute(1, 2, 0)
        return y

class SConv1d(nn.Module):
    """Conv1d with some builtin handling of asymmetric or causal padding
    and normalization.
    """
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, stride: int = 1, dilation: int = 1,
                 groups: int = 1, bias: bool = True, causal: bool = False,
                 norm: str = 'none', norm_kwargs: tp.Dict[str, tp.Any] = {},
                 pad_mode: str = 'reflect'):
        super().__init__()
        # warn user on unusual setup between dilation and stride
        if stride > 1 and dilation > 1:
            warnings.warn('SConv1d has been initialized with stride > 1 and dilation > 1'
                          f' (kernel_size={kernel_size} stride={stride}, dilation={dilation}).')
        self.conv = NormConv1d(in_channels, out_channels, kernel_size, stride,
                               dilation=dilation, groups=groups, bias=bias, causal=causal,
                               norm=norm, norm_kwargs=norm_kwargs)
        self.causal = causal
        self.pad_mode = pad_mode

    def forward(self, x):
        B, C, T = x.shape
        kernel_size = self.conv.conv.kernel_size[0]
        stride = self.conv.conv.stride[0]
        dilation = self.conv.conv.dilation[0]
        kernel_size = (kernel_size - 1) * dilation + 1  # effective kernel size with dilations
        padding_total = kernel_size - stride
        extra_padding = get_extra_padding_for_conv1d(x, kernel_size, stride, padding_total)
        if self.causal:
            # Left padding for causal
            x = pad1d(x, (padding_total, extra_padding), mode=self.pad_mode)
        else:
            # Asymmetric padding required for odd strides
            padding_right = padding_total // 2
            padding_left = padding_total - padding_right
            x = pad1d(x, (padding_left, padding_right + extra_padding), mode=self.pad_mode)
        return self.conv(x)


class SEANetResnetBlock(nn.Module):

    def __init__(self, dim: int, kernel_sizes: tp.List[int] = [3, 1], dilations: tp.List[int] = [1, 1],
                 activation: str = 'ELU', activation_params: dict = {'alpha': 1.0},
                 norm: str = 'weight_norm', norm_params: tp.Dict[str, tp.Any] = {}, causal: bool = False,
                 pad_mode: str = 'reflect', compress: int = 2, true_skip: bool = True):
        super().__init__()
        assert len(kernel_sizes) == len(dilations), 'Number of kernel sizes should match number of dilations'
        act = getattr(nn, activation)
        hidden = dim // compress
        block = []
        for i, (kernel_size, dilation) in enumerate(zip(kernel_sizes, dilations)):
            in_chs = dim if i == 0 else hidden
            out_chs = dim if i == len(kernel_sizes) - 1 else hidden
            block += [
                act(**activation_params),
                SConv1d(in_chs, out_chs, kernel_size=kernel_size, dilation=dilation,
                        norm=norm, norm_kwargs=norm_params,
                        causal=causal, pad_mode=pad_mode),
            ]
        self.block = nn.Sequential(*block)
        self.shortcut: nn.Module
        if true_skip:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = SConv1d(dim, dim, kernel_size=1, norm=norm, norm_kwargs=norm_params,
                                    causal=causal, pad_mode=pad_mode)

    def forward(self, x):
        return self.shortcut(x) + self.block(x)
    
class SEANetEncoder(nn.Module):
    def __init__(self, channels: int = 1, dimension: int = 128, n_filters: int = 32, n_residual_layers: int = 1,
                 ratios: tp.List[int] = [8, 5, 4, 2], activation: str = 'ELU', activation_params: dict = {'alpha': 1.0},
                 norm: str = 'weight_norm', norm_params: tp.Dict[str, tp.Any] = {}, kernel_size: int = 7,
                 last_kernel_size: int = 7, residual_kernel_size: int = 3, dilation_base: int = 2, causal: bool = False,
                 pad_mode: str = 'reflect', true_skip: bool = False, compress: int = 2, lstm: int = 2):
        super().__init__()
        self.channels = channels
        self.dimension = dimension
        self.n_filters = n_filters
        self.ratios = list(reversed(ratios))
        del ratios
        self.n_residual_layers = n_residual_layers
        self.hop_length = np.prod(self.ratios)

        act = getattr(nn, activation)
        mult = 1
        model: tp.List[nn.Module] = [
            SConv1d(channels, mult * n_filters, kernel_size, norm=norm, norm_kwargs=norm_params,
                    causal=causal, pad_mode=pad_mode)
        ]
        # Downsample to raw audio scale
        for i, ratio in enumerate(self.ratios):
            # Add residual layers
            for j in range(n_residual_layers):
                model += [
                    SEANetResnetBlock(mult * n_filters, kernel_sizes=[residual_kernel_size, 1],
                                      dilations=[dilation_base ** j, 1],
                                      norm=norm, norm_params=norm_params,
                                      activation=activation, activation_params=activation_params,
                                      causal=causal, pad_mode=pad_mode, compress=compress, true_skip=true_skip)]

            # Add downsampling layers
            model += [
                act(**activation_params),
                SConv1d(mult * n_filters, mult * n_filters * 2,
                        kernel_size=ratio * 2, stride=ratio,
                        norm=norm, norm_kwargs=norm_params,
                        causal=causal, pad_mode=pad_mode),
            ]
            mult *= 2

        if lstm:
            model += [SLSTM(mult * n_filters, num_layers=lstm)]

        model += [
            act(**activation_params),
            SConv1d(mult * n_filters, dimension, last_kernel_size, norm=norm, norm_kwargs=norm_params,
                    causal=causal, pad_mode=pad_mode)
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

In [15]:
model = SEANetEncoder(channels=1, dimension=128, n_filters=32, n_residual_layers=1,
                 ratios=[8, 5, 4, 2], activation='ELU', activation_params={'alpha': 1.0},
                 norm='weight_norm', norm_params={}, kernel_size=7, last_kernel_size=7, residual_kernel_size=3, dilation_base=2, causal=True,
                 pad_mode='reflect', true_skip=False, compress=2, lstm=2)
x = torch.randn(1, 1, 48000)
emb = model(x)
print(emb.shape)

torch.Size([1, 128, 150])


In [7]:
import torch
from models.slow_branch import Transformer_SlowBranch
from models.decoder import CrossA_Decoder

slow_branch = Transformer_SlowBranch(latent_dim=16, dim=512)
decoder = CrossA_Decoder(in_channels=128, dim=512, shift_dim=320, inter_channels=256, num_blocks=2)

slow_latent = torch.randn(1, 128, 16)
fast_latent = torch.randn(1, 128, 1)

K, V = slow_branch(slow_latent)
print(K.shape, V.shape)
out = decoder(fast_latent, (K, V))

print(out.shape)



torch.Size([1, 128, 512]) torch.Size([1, 128, 512])
torch.Size([1, 320])


/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
